In [ ]:
import sys
from pathlib import Path
for _root in [Path.cwd(), *Path.cwd().parents]:
    if (_root / "paths.py").exists():
        sys.path.insert(0, str(_root))
        break
else:
    raise RuntimeError(
        "Could not find MedGemma-27b-text-it project root (paths.py). Run Jupyter with cwd project root or notebooks/."
    )
import paths


In [3]:
import sys
print(f"Python executable: {sys.executable}")
print(f"Python version: {sys.version}")

import transformers
print(f"Transformers version: {transformers.__version__}")

from transformers import GemmaTokenizer
print("Success! GemmaTokenizer imported.")

Python executable: /home/yuexing/miniconda/bin/python
Python version: 3.13.5 | packaged by Anaconda, Inc. | (main, Jun 12 2025, 16:09:02) [GCC 11.2.0]
Transformers version: 5.0.0
Success! GemmaTokenizer imported.


In [2]:
import transformers
print(f"Transformers version: {transformers.__version__}")
print(f"Location: {transformers.__file__}")

from transformers import GemmaTokenizer
print("GemmaTokenizer imported successfully!")

/home/yuexing/miniconda/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Transformers version: 5.0.0
Location: /home/yuexing/miniconda/lib/python3.13/site-packages/transformers/__init__.py
GemmaTokenizer imported successfully!


In [5]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
import os

os.environ["HF_HOME"] = "/orcd/compute/mghassem/001/gobi1/huggingface"
os.environ["TRANSFORMERS_CACHE"] = "/orcd/compute/mghassem/001/gobi1/huggingface"

# Full path to the model snapshot
model_path = "/orcd/compute/mghassem/001/gobi1/huggingface/hub/models--google--medgemma-27b-text-it/snapshots/5b667cf2ddcf064085bc90952edb35a0edbfb79c"

tokenizer = AutoTokenizer.from_pretrained(
    model_path,
    use_fast=True,
    local_files_only=True
)

model = AutoModelForCausalLM.from_pretrained(
    model_path,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    local_files_only=True
)

prompt = "Give me a short introduction to large language model."

messages = [
    {"role": "user", "content": prompt}
]

text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

generated_ids = model.generate(
    **model_inputs,
    max_new_tokens=2048,
    do_sample=False
)

generated_ids = [
    output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
]

response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
print(response)

Loading weights: 100%|█| 808/808 [00:06<00:00, 117.53it/s, Materializing param=m
Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.


Okay, here's a short introduction to Large Language Models (LLMs):

**Large Language Models (LLMs) are a type of artificial intelligence (AI) designed to understand, generate, and interact with human language.**

Think of them as incredibly sophisticated pattern-matching machines. They are trained on massive amounts of text data (like books, articles, websites) and learn the statistical relationships between words and concepts.

**Key characteristics:**

*   **"Large":** They have billions (or even trillions) of parameters, which are essentially the variables the model adjusts during training to learn patterns.
*   **"Language":** Their primary function is processing and generating human language.
*   **Capabilities:** They can perform tasks like:
    *   Answering questions
    *   Writing essays, code, or creative content
    *   Translating languages
    *   Summarizing text
    *   Holding conversations (like chatbots)

**In essence, LLMs are powerful tools that can mimic human-lik

In [ ]:
import pandas as pd 
import re
import torch
import os

# Load data
df = pd.read_csv(paths.DATA / "MedGemma_SR_Match_Rate.csv")
print("Columns in dataset:")
print(df.columns.tolist())

# Function to extract answer letter using multiple patterns
def extract_answer_letter(text):
    if pd.isna(text) or not text:
        return None
    
    # Try different patterns to extract the answer letter
    patterns = [
        r"Answer:\s*([A-J])",             # "Answer: A"
        r"Answer is\s*([A-J])",           # "Answer is A"
        r"answer is\s*([A-J])",           # "answer is A"
        r"The answer is\s*([A-J])",       # "The answer is A"
        r"the answer is\s*([A-J])",       # "the answer is A"
        r"Option\s*([A-J])",              # "Option A"
        r"option\s*([A-J])",              # "option A"
        r"My answer is\s*([A-J])",        # "My answer is A"
        r"(\n|^)([A-J])\.?\s*$",          # "A." or just "A" at end or newline
        r"select option\s*([A-J])",       # "select option A"
        r"I select\s*([A-J])",            # "I select A"
        r"I choose\s*([A-J])",            # "I choose A"
    ]
    
    for pattern in patterns:
        match = re.search(pattern, text)
        if match:
            # Some patterns have the letter in group 1, others in group 2
            return match.group(1) if len(match.groups()) == 1 else match.group(2)
    
    # If no match found, check if there's a single letter at the end
    words = text.strip().split()
    if words and len(words[-1]) == 1 and words[-1].isalpha() and words[-1].upper() in "ABCDEFGHIJ":
        return words[-1].upper()
    
    return None


# At the beginning, before the loop
progress_file = paths.PREDICTIONS / "MedGemma27B_predictions_Gemma_SR_progress.csv"

# Check if progress file exists and load it
if os.path.exists(progress_file):
    existing_results = pd.read_csv(progress_file)
    # Extract processed row indices from QA_ID (format: "Merge Q123")
    processed_indices = set()
    for qa_id in existing_results['QA_ID']:
        # Extract number from "Merge Q123" -> 123, then convert to 0-indexed (122)
        idx = int(qa_id.split('Q')[1]) - 1
        processed_indices.add(idx)
    
    results = existing_results.to_dict('records')
    print(f"Found {len(processed_indices)} already processed rows. Resuming...")
else:
    processed_indices = set()
    results = []
    print("Starting from scratch...")


# Loop through the dataset
total_rows = len(df)
print(f"Processing {total_rows} rows...")

for idx, row in df.head(total_rows).iterrows():
    # Skip if already processed
    if idx in processed_indices:
        print(f"Skipping row {idx+1}/{total_rows} (already processed)...")
        continue
    
    print(f"Processing row {idx+1}/{total_rows}...")
    try:
        context_text = row["MedGemma_High_Relevance"]
        question = row["question_options_x"]
        
        # Improved prompt with clearer instructions
        prompt = (
            "You are a clinical reasoning assistant. You will receive a patient case summary "
            "and a multiple-choice question.\n\n"
            f"{context_text}\n\n"
            f"{question}\n\n"
            "Please select the single most appropriate answer. Respond only in the following format:\n\n"
            "Answer: <LETTER>"
        )
    
        # Use chat template format (like your working example)
        messages = [
            {"role": "system", "content": "You are MedGemma. You are a helpful medical assistant."},
            {"role": "user", "content": prompt}
        ]
        
        text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )
        
        model_inputs = tokenizer([text], return_tensors="pt").to(model.device)
        
        # Generate prediction
        with torch.no_grad():
            generated_ids = model.generate(
                **model_inputs,
                max_new_tokens=128,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id
            )
        
        # Extract only the generated part (not the input)
        generated_ids = [
            output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
        ]
        
        # Decode the response
        raw_response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
        
        # Extract the answer letter
        extracted_answer = extract_answer_letter(raw_response)
        
        
        # If still no answer found, log more details for debugging
        if extracted_answer is None:
            print(f"⚠️ Could not extract answer from response for row {idx+1}:")
            print(f"Response: {raw_response[:100]}...")
        
        # Create result entry
        qa_id = f"Merge Q{idx + 1}"
        result_entry = {
            "QA_ID": qa_id,
            "Origin": row.get("Origin", ""),
            "data_source": row.get("data_source_corr", ""),
            "Raw_Response": raw_response,
            "Extracted_Answer": extracted_answer
        }
        
        results.append(result_entry)
        processed_indices.add(idx)
        print(f"✅ Processed {qa_id}: Answer = {extracted_answer}")
        
        # Save progress every 10 items (increased frequency for safety)
        if (idx + 1) % 10 == 0:
            temp_df = pd.DataFrame(results)
            temp_df.to_csv(progress_file, index=False)
            print(f"Saved progress to CSV after {idx+1} items")

    except Exception as e:
        print(f"❌ Error on row {idx}: {str(e)}")
        # Still try to save the entry with error info
        qa_id = f"Merge Q{idx + 1}"
        results.append({
            "QA_ID": qa_id,
            "Origin": row.get("Origin", ""),
            "data_source": row.get("data_source_corr", ""),
            "Raw_Response": f"ERROR: {str(e)}",
            "Extracted_Answer": None
        })
        processed_indices.add(idx)

# Save final results
output_df = pd.DataFrame(results)
output_file = paths.PREDICTIONS / "MedGemma27B_predictions_Gemma_SR.csv"
output_df.to_csv(output_file, index=False)
print(f"Saved all predictions to {output_file}")

Columns in dataset:
['Origin', 'data_source_df3', 'Patient_Profile', 'Low+Irr', 'High', 'question_options_x', 'answer_corr', 'ID', 'centaur_question', 'sentence_number', 'answer', 'data_source', 'step1_excerpts', 'question_options_y', 'step1_sentences', 'sentence_1', 'sentence_2', 'sentence_3', 'sentence_4', 'sentence_5', 'sentence_6', 'sentence_7', 'sentence_8', 'sentence_9', 'sentence_10', 'sentence_11', 'sentence_12', 'sentence_13', 'sentence_14', 'sentence_15', 'sentence_16', 'sentence_17', 'sentence_18', 'sentence_19', 'sentence_20', 'sentence_21', 'non_none_sentence_count', 'MedGemma27B_answer', 'MedGemma27B_raw_response', 'q1', 'q2', 'q3', 'q4', 'q5', 'q6', 'q7', 'q8', 'q9', 'q10', 'q11', 'q12', 'q13', 'q14', 'q15', 'q16', 'q17', 'q18', 'q19', 'q20', 'label_21', 'MedGemma_High_Relevance', 'Match?', 'data_source_corr_trainee', 'Match_percent']
Starting from scratch...
Processing 1300 rows...
Processing row 1/1300...
⚠️ Could not extract answer from response for row 1:
Response: <

✅ Processed Merge Q49: Answer = B
Processing row 50/1300...
✅ Processed Merge Q50: Answer = A
Saved progress to CSV after 50 items
Processing row 51/1300...
⚠️ Could not extract answer from response for row 51:
Response: <unused94>thought
The user wants me to identify the most likely cause of the patient's symptoms base...
✅ Processed Merge Q51: Answer = None
Processing row 52/1300...
⚠️ Could not extract answer from response for row 52:
Response: <unused94>thought
The user wants me to identify the physical exam finding most strongly associated w...
✅ Processed Merge Q52: Answer = None
Processing row 53/1300...
✅ Processed Merge Q53: Answer = C
Processing row 54/1300...
✅ Processed Merge Q54: Answer = B
Processing row 55/1300...
✅ Processed Merge Q55: Answer = B
Processing row 56/1300...
⚠️ Could not extract answer from response for row 56:
Response: <unused94>thought
The user wants me to identify the best initial therapy for a patient presenting wi...
✅ Processed Merge Q56: Answer = N

✅ Processed Merge Q116: Answer = A
Processing row 117/1300...
✅ Processed Merge Q117: Answer = A
Processing row 118/1300...
✅ Processed Merge Q118: Answer = D
Processing row 119/1300...
✅ Processed Merge Q119: Answer = C
Processing row 120/1300...
✅ Processed Merge Q120: Answer = A
Saved progress to CSV after 120 items
Processing row 121/1300...
✅ Processed Merge Q121: Answer = B
Processing row 122/1300...
⚠️ Could not extract answer from response for row 122:
Response: <unused94>thought
The user wants me to identify the most likely diagnosis based on the provided pati...
✅ Processed Merge Q122: Answer = None
Processing row 123/1300...
⚠️ Could not extract answer from response for row 123:
Response: <unused94>thought
The user wants me to analyze a clinical case and choose the most appropriate next ...
✅ Processed Merge Q123: Answer = None
Processing row 124/1300...
⚠️ Could not extract answer from response for row 124:
Response: <unused94>thought
The patient is a 41-year-old woman with

✅ Processed Merge Q173: Answer = C
Processing row 174/1300...
✅ Processed Merge Q174: Answer = B
Processing row 175/1300...
⚠️ Could not extract answer from response for row 175:
Response: <unused94>thought
The user wants me to identify the most likely additional finding in a patient with...
✅ Processed Merge Q175: Answer = None
Processing row 176/1300...
⚠️ Could not extract answer from response for row 176:
Response: <unused94>thought
The patient is a 24-year-old primigravida at 31 weeks' gestation presenting with e...
✅ Processed Merge Q176: Answer = None
Processing row 177/1300...
⚠️ Could not extract answer from response for row 177:
Response: <unused94>thought
The user wants me to identify the most appropriate endotracheal tube (ETT) size fo...
✅ Processed Merge Q177: Answer = None
Processing row 178/1300...
✅ Processed Merge Q178: Answer = A
Processing row 179/1300...
✅ Processed Merge Q179: Answer = B
Processing row 180/1300...
✅ Processed Merge Q180: Answer = A
Saved progress 

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats

# Load the data
output_df = pd.read_csv(paths.PREDICTIONS / "MedGemma27B_predictions_Gemma_SR.csv")
df = pd.read_csv(paths.DATA / "gpt5-irr-removed-relevancy-combined-dec-12.csv")

# Ensure both dataframes have the same length
assert len(output_df) == len(df), "DataFrames have different lengths!"

# Calculate None/Empty cells in Extracted_Answer - OVERALL
total_empty_cells = output_df['Extracted_Answer'].isna().sum() + (output_df['Extracted_Answer'] == '').sum()
total_cells = len(output_df)
empty_percentage = (total_empty_cells / total_cells) * 100

print("=" * 60)
print("NONE/EMPTY CELL ANALYSIS - OVERALL")
print("=" * 60)
print(f"Total None/Empty cells in 'Extracted_Answer': {total_empty_cells}")
print(f"Total cells: {total_cells}")
print(f"Percentage None/Empty: {empty_percentage:.2f}%")
print("=" * 60)
print()

# Calculate accuracy (assuming both columns contain the same type of answers to compare)
# Method 1: Exact match
output_df['match'] = (output_df['Extracted_Answer'] == df['answer_corr']).astype(int)

# Overall accuracy statistics
accuracy = output_df['match'].mean()
std_dev = output_df['match'].std()
n = len(output_df)
se = std_dev / np.sqrt(n)  # Standard error
ci_95 = stats.t.interval(0.95, n-1, loc=accuracy, scale=se)

print("=" * 60)
print("OVERALL ACCURACY ANALYSIS")
print("=" * 60)
print(f"Accuracy: {accuracy:.4f} ({accuracy*100:.2f}%)")
print(f"Standard Deviation: {std_dev:.4f}")
print(f"95% Confidence Interval: [{ci_95[0]:.4f}, {ci_95[1]:.4f}]")
print(f"95% CI (percentage): [{ci_95[0]*100:.2f}%, {ci_95[1]*100:.2f}%]")
print(f"Sample Size: {n}")
print("=" * 60)
print()

# Analysis by category (assuming data_source_corr is in one of the dataframes)
# Check which dataframe has data_source_corr
if 'data_source_corr' in output_df.columns:
    analysis_df = output_df.copy()
elif 'data_source_corr' in df.columns:
    analysis_df = output_df.copy()
    analysis_df['data_source_corr'] = df['data_source_corr']
else:
    print("Warning: 'data_source_corr' column not found in either dataframe")
    analysis_df = output_df.copy()

# Category-wise analysis
if 'data_source_corr' in analysis_df.columns:
    # Calculate None/Empty cells by data_source
    empty_by_source = analysis_df.groupby('data_source_corr').apply(
        lambda x: pd.Series({
            'None_Count': x['Extracted_Answer'].isna().sum(),
            'Empty_String_Count': (x['Extracted_Answer'] == '').sum(),
            'Total_Empty': x['Extracted_Answer'].isna().sum() + (x['Extracted_Answer'] == '').sum()
        })
    ).reset_index()
    
    source_totals = analysis_df.groupby('data_source_corr').size().reset_index(name='Total_Count')
    empty_summary = empty_by_source.merge(source_totals, on='data_source_corr')
    empty_summary['Empty_Percentage'] = (empty_summary['Total_Empty'] / empty_summary['Total_Count']) * 100
    
    print("NONE/EMPTY CELLS BY DATA SOURCE")
    print("=" * 60)
    print(empty_summary.to_string(index=False))
    print("=" * 60)
    print()
    
    category_stats = analysis_df.groupby('data_source_corr')['match'].agg([
        ('Count', 'count'),
        ('Mean_Accuracy', 'mean'),
        ('Std_Dev', 'std'),
        ('SE', lambda x: x.std() / np.sqrt(len(x)))
    ]).reset_index()
    
    # Calculate 95% CI for each category
    ci_lower = []
    ci_upper = []
    
    for idx, row in category_stats.iterrows():
        n_cat = row['Count']
        mean_cat = row['Mean_Accuracy']
        se_cat = row['SE']
        
        if n_cat > 1:
            ci = stats.t.interval(0.95, n_cat-1, loc=mean_cat, scale=se_cat)
            ci_lower.append(ci[0])
            ci_upper.append(ci[1])
        else:
            ci_lower.append(np.nan)
            ci_upper.append(np.nan)
    
    category_stats['CI_95_Lower'] = ci_lower
    category_stats['CI_95_Upper'] = ci_upper
    
    # Format percentages
    category_stats['Mean_Accuracy_%'] = category_stats['Mean_Accuracy'] * 100
    category_stats['Std_Dev_%'] = category_stats['Std_Dev'] * 100
    category_stats['CI_95_Lower_%'] = category_stats['CI_95_Lower'] * 100
    category_stats['CI_95_Upper_%'] = category_stats['CI_95_Upper'] * 100
    
    print("CATEGORY-WISE ACCURACY ANALYSIS")
    print("=" * 60)
    print(category_stats.to_string(index=False))
    print("=" * 60)
    print()
    
# Create summary statistics table
summary_table = pd.DataFrame({
    'Metric': ['Overall Accuracy', 'Standard Deviation', '95% CI Lower', '95% CI Upper', 'Sample Size'],
    'Value': [f"{accuracy:.4f} ({accuracy*100:.2f}%)", 
              f"{std_dev:.4f}", 
              f"{ci_95[0]:.4f} ({ci_95[0]*100:.2f}%)", 
              f"{ci_95[1]:.4f} ({ci_95[1]*100:.2f}%)", 
              n]
})

print("\nSUMMARY TABLE")
print("=" * 60)
print(summary_table.to_string(index=False))
print("=" * 60)